In [0]:
bronze_path = "/Volumes/parisolympics/dbo/bronze/"
silver_path = 'parisolympics.silver/'

In [0]:
athletes = 'athletes.csv'
nocs = 'nocs.csv'

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [0]:
athletes_df = spark.read.format("csv").option("inferSchema", "True").option("header", "True").load(bronze_path + athletes)
nocs_df = spark.read.format("csv").option("inferSchema", "True").option("header", "True").load(bronze_path + nocs)

In [0]:
nocs_df = nocs_df.drop("country")

In [0]:
nocs_df = nocs_df.withColumn("tag" , split(col("tag"), "-")[0])

In [0]:
nocs_df.write.mode("overwrite").format("delta").option("path" , "/Volumes/parisolympics/dbo/silver/nocs").save()

In [0]:
athletes_df.display()

In [0]:
df_filter = athletes_df.fillna({"birth_place": "xyz" , "birth_country": "abc", "residence_place": "unkown", "residence_country": "aaa"})\
                            .filter((col("name").isin('GALSTYAN Slavik' , 'HARUTYUNYAN Arsen', 'SEHEN Sajjad')) & (col("current") == 'true') )

In [0]:
df = athletes_df.withColumn("height",col("height").cast("float"))\
            .withColumn("weight",col("weight").cast("float"))

In [0]:
df = df.orderBy(col("height").desc() , col("weight"))\
    .filter(col("weight")>0)

In [0]:
df = df.withColumn("nationality", regexp_replace(col("nationality"), "United States", "US" ))

In [0]:
df = df.withColumnRenamed("code","athlete_id")

In [0]:
df.display()


In [0]:
df = df.withColumn("occupation" , split(col("occupation"), ","))

In [0]:
df.display()

In [0]:
df =df.select('athlete_id',
 'current',
 'name',
 'name_short',
 'name_tv',
 'gender',
 'function',
 'country_code',
 'country',
 'country_long',
 'nationality',
 'nationality_long',
 'nationality_code',
 'height',
 'weight')

In [0]:
df.display()

In [0]:
df.write.mode("overwrite").format("delta").option("path" , "/Volumes/parisolympics/dbo/silver/athlete").save()